# 02. π0 openpi + V-JEPA 코드 구조 탐색

**모듈**: M4 - Architecture Analysis
**날짜**: 2026-03-11

실제 구현 코드의 구조를 분석하여 우리 프로젝트에 적용할 요소를 파악합니다.

**분석 대상**:
- π0 openpi: https://github.com/Physical-Intelligence/openpi
- pi-zero-pytorch: https://github.com/lucidrains/pi-zero-pytorch
- V-JEPA: https://github.com/facebookresearch/jepa

In [ ]:
import torch
import torch.nn as nn
import numpy as np
print(f'PyTorch: {torch.__version__}')
print(f'MPS 사용 가능: {torch.backends.mps.is_available()}')

## 1. π0 openpi 코드 구조 분석

### 핵심 파일 구조 (공식 repo 기준)

```
openpi/
├── src/openpi/
│   ├── models/
│   │   ├── pi0.py              # 메인 모델 (VLM + Action Expert)
│   │   ├── tokenizer.py        # 행동 토큰화
│   │   └── flow_matching.py    # Flow Matching 헤드
│   ├── training/
│   │   ├── config.py           # 학습 설정
│   │   └── train.py            # 학습 루프
│   └── shared/
│       └── normalize.py        # 행동 정규화
└── examples/
    └── simple/                  # 간단한 사용 예시
```

### 핵심 코드 패턴 분석

In [ ]:
# π0의 핵심 패턴을 간소화하여 재현
# (실제 openpi는 PaliGemma VLM을 사용하지만, 우리는 간단한 Encoder로 대체)

class SimplifiedPi0(nn.Module):
    """
    π0의 핵심 구조를 간소화한 버전
    원본: PaliGemma VLM (3B) + Action Expert + Flow Matching
    우리: 작은 Encoder + MLP Action Head + Flow Matching
    """
    
    def __init__(self, obs_dim=64, action_dim=2, hidden_dim=128):
        super().__init__()
        
        # π0의 VLM 역할: 관측 인코딩 (우리는 간단한 MLP)
        self.state_encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        
        # π0의 Action Expert 역할: 상태 → 행동 생성
        # Flow Matching: 시간 t + 노이즈 행동 → 속도장 예측
        self.flow_net = nn.Sequential(
            nn.Linear(hidden_dim + action_dim + 1, hidden_dim),  # state + noisy_action + t
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),  # 속도장 (velocity field)
        )
        
        self.action_dim = action_dim
    
    def compute_loss(self, obs, target_actions):
        """Flow Matching 학습 (핵심!)"""
        batch_size = obs.shape[0]
        state = self.state_encoder(obs)
        
        # 1. 랜덤 시간 t ∈ [0, 1]
        t = torch.rand(batch_size, 1)
        
        # 2. 노이즈 (시작점)
        noise = torch.randn_like(target_actions)
        
        # 3. 선형 보간: x_t = (1-t) * noise + t * target
        x_t = (1 - t) * noise + t * target_actions
        
        # 4. 진짜 속도장: v = target - noise
        target_velocity = target_actions - noise
        
        # 5. 네트워크가 속도장 예측
        flow_input = torch.cat([state, x_t, t], dim=-1)
        pred_velocity = self.flow_net(flow_input)
        
        # 6. MSE Loss
        loss = nn.functional.mse_loss(pred_velocity, target_velocity)
        return loss
    
    @torch.no_grad()
    def generate_action(self, obs, n_steps=20):
        """Flow Matching 추론 (노이즈 → 행동)"""
        state = self.state_encoder(obs)
        
        # 노이즈에서 시작
        action = torch.randn(1, self.action_dim)
        dt = 1.0 / n_steps
        
        # 직선 경로를 따라 이동
        for i in range(n_steps):
            t = torch.tensor([[i / n_steps]])
            flow_input = torch.cat([state, action, t], dim=-1)
            velocity = self.flow_net(flow_input)
            action = action + velocity * dt
        
        return action

# 테스트
model = SimplifiedPi0(obs_dim=64, action_dim=2)
print(f'모델 파라미터: {sum(p.numel() for p in model.parameters()):,}')

# 가상 데이터로 학습 테스트
obs = torch.randn(8, 64)  # batch=8, obs_dim=64
actions = torch.randn(8, 2)  # batch=8, action_dim=2

loss = model.compute_loss(obs, actions)
print(f'학습 loss: {loss.item():.4f}')

# 추론 테스트
test_obs = torch.randn(1, 64)
generated = model.generate_action(test_obs, n_steps=20)
print(f'생성된 행동: {generated.numpy().flatten()}')

## 2. V-JEPA 코드 구조 분석

### 핵심 파일 구조 (facebookresearch/jepa)

```
jepa/
├── src/
│   ├── models/
│   │   ├── vision_transformer.py   # ViT Encoder
│   │   └── predictor.py            # Predictor (좁은 Transformer)
│   ├── masks/
│   │   └── multiblock.py           # 시공간 마스킹 전략
│   └── train.py                     # EMA + 학습 루프
└── configs/
    └── pretrain/
        └── vitl16.yaml              # ViT-Large 설정
```

### 핵심 코드 패턴 분석

In [ ]:
# V-JEPA의 핵심 메커니즘을 간소화하여 재현

class SimplifiedVJEPA(nn.Module):
    """
    V-JEPA의 핵심 구조를 간소화한 버전
    원본: ViT-Large + 시공간 마스킹 + EMA Target Encoder
    우리: 작은 MLP Encoder + 시간 마스킹 + EMA
    """
    
    def __init__(self, input_dim=64, hidden_dim=128, latent_dim=64, ema_decay=0.996):
        super().__init__()
        self.ema_decay = ema_decay
        
        # Context Encoder (학습됨)
        self.context_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        
        # Target Encoder (EMA 업데이트, gradient 없음)
        self.target_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        # 초기값: Context Encoder와 동일
        self._copy_params(self.context_encoder, self.target_encoder)
        
        # Predictor (보이는 표현 → 가려진 표현 예측)
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),  # latent + position encoding
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
    
    def _copy_params(self, src, dst):
        for p_src, p_dst in zip(src.parameters(), dst.parameters()):
            p_dst.data.copy_(p_src.data)
            p_dst.requires_grad = False
    
    @torch.no_grad()
    def _ema_update(self):
        """Target Encoder를 Context Encoder의 EMA로 업데이트"""
        for p_ctx, p_tgt in zip(self.context_encoder.parameters(),
                                  self.target_encoder.parameters()):
            p_tgt.data = self.ema_decay * p_tgt.data + (1 - self.ema_decay) * p_ctx.data
    
    def forward(self, sequence):
        """
        sequence: (batch, seq_len, input_dim) — 시간 순서대로 센서 데이터
        마스킹: 마지막 N개 프레임을 예측 대상으로
        """
        batch_size, seq_len, _ = sequence.shape
        context_len = seq_len - 1  # 마지막 1개를 예측
        
        # Context: 보이는 프레임 인코딩
        context = sequence[:, :context_len]  # (batch, context_len, input_dim)
        context_features = self.context_encoder(context)  # (batch, context_len, latent_dim)
        # 평균 풀링으로 요약
        context_summary = context_features.mean(dim=1)  # (batch, latent_dim)
        
        # Target: 가려진 프레임의 "정답" (gradient 없음)
        with torch.no_grad():
            target = sequence[:, context_len:]  # (batch, 1, input_dim)
            target_features = self.target_encoder(target)  # (batch, 1, latent_dim)
            target_features = target_features.squeeze(1)  # (batch, latent_dim)
        
        # Predictor: context → 가려진 부분 예측
        # 위치 정보 추가 (어떤 시간의 표현을 예측하는지)
        pos = torch.tensor([[context_len / seq_len]]).expand(batch_size, 1)
        pred_input = torch.cat([context_summary, pos], dim=-1)
        predicted_features = self.predictor(pred_input)  # (batch, latent_dim)
        
        # Loss: 잠재공간에서의 L2 거리
        loss = nn.functional.mse_loss(predicted_features, target_features)
        
        # EMA 업데이트
        self._ema_update()
        
        return loss, predicted_features, target_features
    
    @torch.no_grad()
    def predict_future(self, context_sequence):
        """학습 후: 보이는 시퀀스로 미래 표현 예측"""
        context_features = self.context_encoder(context_sequence)
        context_summary = context_features.mean(dim=1)
        seq_len = context_sequence.shape[1]
        pos = torch.tensor([[1.0]])  # 다음 시점
        pred_input = torch.cat([context_summary, pos], dim=-1)
        return self.predictor(pred_input)

# 테스트
vjepa = SimplifiedVJEPA(input_dim=64, latent_dim=64)
print(f'V-JEPA 파라미터: {sum(p.numel() for p in vjepa.parameters()):,}')

# 시퀀스 데이터 (5 스텝)
sequence = torch.randn(4, 5, 64)  # batch=4, seq=5, dim=64

loss, pred, target = vjepa(sequence)
print(f'V-JEPA loss: {loss.item():.4f}')
print(f'예측 shape: {pred.shape}, 타겟 shape: {target.shape}')

# 미래 예측 테스트
context = torch.randn(1, 3, 64)  # 3스텝 관측
future = vjepa.predict_future(context)
print(f'미래 예측: shape={future.shape}')

## 3. 차용 vs 자체 구현 정리

In [ ]:
print('=== 차용 포인트 정리 ===')
print()
print('┌──────────────────────────────────────────────────────────────────┐')
print('│ π0에서 차용                                                      │')
print('├──────────────────────────────────────────────────────────────────┤')
print('│ ✅ Flow Matching 학습/추론 패턴 (noise → action via velocity)   │')
print('│ ✅ 상태 인코딩과 행동 생성 분리 구조                              │')
print('│ ✅ 조건부 생성 (context → 행동)                                  │')
print('│ ❌ PaliGemma VLM (너무 큼) → 작은 Encoder로 대체                │')
print('│ ❌ 범용 로봇 제어 → 장애물 회피 특화                              │')
print('└──────────────────────────────────────────────────────────────────┘')
print()
print('┌──────────────────────────────────────────────────────────────────┐')
print('│ V-JEPA에서 차용                                                  │')
print('├──────────────────────────────────────────────────────────────────┤')
print('│ ✅ 잠재공간에서의 미래 예측 (픽셀 복원 없음)                       │')
print('│ ✅ EMA Target Encoder (안정적 학습)                              │')
print('│ ✅ Context → Predictor → 미래 표현 구조                          │')
print('│ ✅ 시간적 마스킹 기반 학습                                        │')
print('│ ❌ ViT-Large (너무 큼) → 작은 MLP/Transformer                   │')
print('│ ❌ 대규모 비디오 사전학습 → 환경 에피소드에서 학습                  │')
print('└──────────────────────────────────────────────────────────────────┘')
print()
print('┌──────────────────────────────────────────────────────────────────┐')
print('│ 우리만의 추가                                                     │')
print('├──────────────────────────────────────────────────────────────────┤')
print('│ 🆕 5개 센서 멀티모달 Fusion                                      │')
print('│ 🆕 LLM 양방향 센서 (상황해석 + 명령수신)                          │')
print('│ 🆕 V-JEPA + LLM = 이중 설명 가능성                               │')
print('│ 🆕 M3 ObstacleAvoidanceEnv 특화 환경                             │')
print('└──────────────────────────────────────────────────────────────────┘')

## 4. End-to-End 파이프라인 프로토타입

In [ ]:
# 전체 파이프라인을 최소 코드로 연결해보기
# (실제 구현은 M5~M8에서 진행, 여기서는 구조만 확인)

class MiniTripleIntegration(nn.Module):
    """Triple Integration 최소 프로토타입 (~30줄)"""
    
    def __init__(self):
        super().__init__()
        # M5: 5개 Encoder (각 센서 → 16차원)
        self.camera_enc = nn.Linear(3*8*8, 16)   # 단순화: 3x8x8
        self.dist_enc   = nn.Linear(8, 16)
        self.audio_enc  = nn.Linear(2*10, 16)    # 단순화: 2x10
        self.imu_enc    = nn.Linear(6, 16)
        self.lang_enc   = nn.Linear(32, 16)      # 단순화: 32차원 임베딩
        
        # M6: Fusion (5x16=80 → 64)
        self.fusion = nn.Linear(80, 64)
        
        # M7: V-JEPA (64 → 64, 미래 예측)
        self.vjepa_predictor = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64)
        )
        
        # M8: Flow Matching (64+64 → 2, 행동)
        self.flow_net = nn.Sequential(
            nn.Linear(128+2+1, 64), nn.ReLU(), nn.Linear(64, 2)  # +action+t
        )
    
    def forward(self, camera, distance, audio, imu, lang):
        # M5: Encode
        z_cam  = self.camera_enc(camera.flatten(1))
        z_dist = self.dist_enc(distance)
        z_aud  = self.audio_enc(audio.flatten(1))
        z_imu  = self.imu_enc(imu)
        z_lang = self.lang_enc(lang)
        
        # M6: Fuse
        z_all = torch.cat([z_cam, z_dist, z_aud, z_imu, z_lang], dim=-1)  # (80,)
        z_fused = self.fusion(z_all)  # (64,)
        
        # M7: Predict future
        z_future = self.vjepa_predictor(z_fused)  # (64,)
        
        # M8: Generate action (1 step of flow matching)
        noise_action = torch.randn(camera.shape[0], 2)
        t = torch.zeros(camera.shape[0], 1)
        fm_input = torch.cat([z_fused, z_future, noise_action, t], dim=-1)
        action = self.flow_net(fm_input)
        
        return action, z_fused, z_future

# 테스트
model = MiniTripleIntegration()
total_params = sum(p.numel() for p in model.parameters())
print(f'Mini Triple Integration 파라미터: {total_params:,}')

# 가상 센서 데이터
batch = 4
camera = torch.randn(batch, 3, 8, 8)
distance = torch.randn(batch, 8)
audio = torch.randn(batch, 2, 10)
imu = torch.randn(batch, 6)
lang = torch.randn(batch, 32)

action, z_fused, z_future = model(camera, distance, audio, imu, lang)
print(f'\n입력:')
print(f'  camera: {camera.shape}')
print(f'  distance: {distance.shape}')
print(f'  audio: {audio.shape}')
print(f'  imu: {imu.shape}')
print(f'  lang: {lang.shape}')
print(f'\n출력:')
print(f'  action: {action.shape} = {action[0].detach().numpy()}')
print(f'  z_fused: {z_fused.shape} (현재 상태 잠재 벡터)')
print(f'  z_future: {z_future.shape} (V-JEPA 미래 예측)')
print(f'\n전체 파이프라인이 {total_params:,}개 파라미터로 동작!')
print(f'(π0 원본: ~3,000,000,000 → 우리: {total_params:,} = {3_000_000_000/total_params:.0f}배 작음)')

## 5. 정리

In [ ]:
print('=== 실습 2 정리 ===')
print()
print('1. π0 openpi: VLM + Action Expert + Flow Matching 구조 확인')
print('   → Flow Matching 학습/추론 패턴 재현 (SimplifiedPi0)')
print()
print('2. V-JEPA: Context Encoder + EMA Target + Predictor 구조 확인')
print('   → 시간적 마스킹 + 잠재공간 예측 재현 (SimplifiedVJEPA)')
print()
print('3. 차용 포인트: Flow Matching 패턴, 잠재공간 예측, EMA')
print('   자체 추가: 5센서 멀티모달, LLM 센서, 이중 설명 가능성')
print()
print('4. End-to-End 프로토타입: ~30줄, ~25K 파라미터로 전체 구조 검증')
print()
print('다음: 실습 3 - Triple Integration 아키텍처 설계 문서')